In [16]:
import pandas as pd
import lightning.pytorch as pl
import torch.nn as nn
import torch.nn.functional as F
import torch
import lime
import os
import numpy as np
import matplotlib.pyplot as plt

from skimage.segmentation import mark_boundaries
from lime import lime_image
from collections import defaultdict
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateFinder, BatchSizeFinder, EarlyStopping

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image, ImageFilter

from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

# Inicio Práctica

In [17]:
# cassava_df = pd.read_csv('/home/alumno/Desktop/datos/Computer Vision/cassava/cassava_split.csv')
cassava_df = pd.read_csv('/home/alumno/Desktop/datos/cassava/cassava_split.csv')
IMAGE_ROOT_DIR = '/home/alumno/Desktop/datos/cassava/images/'

In [18]:
#  1. CLASE CASSAVADATASET 
class CassavaDataset(Dataset):
    """
    Dataset personalizado para el conjunto de datos de Cassava.
    """
    def __init__(self, df: pd.DataFrame, root_dir: str, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_id']
        label = int(row['label']) 
        
        img_path = os.path.join(self.root_dir, img_name)
        # Cargamos la imagen y aseguramos el formato RGB
        image = Image.open(img_path).convert('RGB') 
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)


In [19]:
class CassavaDataModule(pl.LightningDataModule):
    def __init__(self, data_df: pd.DataFrame, image_dir: str, batch_size: int = 64, train_transform=None): #  Renombrado para claridad
        
        super().__init__()
        self.data_df = data_df
        self.image_dir = image_dir
        self.batch_size = batch_size
        
        # 2: Lógica de transformaciones separada
        
        # 2.1. Define la transformación de validación y test (la que antes era 'por defecto')
        # Esta transformación NO lleva data augmentation.
        self.val_test_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
        ])
        
        # 2.2. Asigna la transformación de entrenamiento
        if train_transform is None:
            # Si el usuario no pasa una 'train_transform', 
            # usamos la misma de validación (sin augmentation)
            self.train_transform = self.val_test_transform
            print("INFO: No se proporcionó 'train_transform'. Usando transformaciones de validación para train.")
        else:
            # Si el usuario SÍ pasa una 'train_transform', la usamos.
            self.train_transform = train_transform
            print("INFO: Usando 'train_transform' personalizada para el conjunto de entrenamiento.")


    def setup(self, stage: str = None): # <-- 'stage' puede ser None por defecto
        # 1. Filtrar el DataFrame basado en la columna 'set'
        train_df = self.data_df[self.data_df['set'] == 'train'].reset_index(drop=True)
        val_df = self.data_df[self.data_df['set'] == 'val'].reset_index(drop=True)
        test_df = self.data_df[self.data_df['set'] == 'test'].reset_index(drop=True)
        
        # Asignar las transformaciones correctas a cada Dataset 
        self.train_ds = CassavaDataset(train_df, self.image_dir, 
                                       transform=self.train_transform) # <-- Transformación de TRAIN
        
        self.val_ds = CassavaDataset(val_df, self.image_dir, 
                                     transform=self.val_test_transform) # <-- Transformación de VAL/TEST
        
        self.test_ds = CassavaDataset(test_df, self.image_dir, 
                                      transform=self.val_test_transform) # <-- Transformación de VAL/TEST

        print(f"Dataset sizes: Train={len(self.train_ds)}, Val={len(self.val_ds)}, Test={len(self.test_ds)}")
        print(f"Train Transforms: {self.train_transform}")
        print(f"Val/Test Transforms: {self.val_test_transform}")

    #  Métodos Dataloader  
    def train_dataloader(self):
        labels = self.train_ds.df['label'].values
        class_sample_count = np.array([len(np.where(labels == t)[0]) for t in np.unique(labels)])
        weights = 1. / class_sample_count
        samples_weight = np.array([weights[t] for t in labels])
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(samples_weight),
            num_samples=len(samples_weight),
            replacement=True
        )
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            sampler=sampler,
            num_workers=4
        )

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers = 4)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers = 4)

In [20]:
class LightningModule(pl.LightningModule):
    def __init__(self, model, lr=1e-3, wd=0., discriminative_lr=None):
        super().__init__()
        self.model = model
        self.lr = lr
        self.wd = wd
        self.discriminative_lr = discriminative_lr # (lr, lr_mult)

        self.training_step_outputs = defaultdict(float)
        self.validation_step_outputs = defaultdict(float)

    def get_layer_wise_lr(self, lr, lr_mult):

        # Save layer names
        layer_names = []
        for idx, (name, param) in enumerate(self.model.named_parameters()):
            layer_names.append(name)
            print(f'{idx}: {name}')
        
        # Reverse layers
        layer_names.reverse()
        
        # placeholder
        parameters      = []
        prev_group_name = layer_names[0].split('.')[0]
        
        # store params & learning rates
        for idx, name in enumerate(layer_names):
            
            # parameter group name
            cur_group_name = name.split('.')[0]
            
            # update learning rate
            if cur_group_name != prev_group_name:
                lr *= lr_mult
            prev_group_name = cur_group_name
            
            # display info
            print('Using discriminative learning rates')
            print(f'{idx}: lr = {lr:.6f}, {name}')
            
            # append layer parameters
            parameters += [{'params': [p for n, p in model.named_parameters() if n == name and p.requires_grad],
                            'lr':     lr}]

        return parameters
        

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        
        return torch.optim.Adam(
            self.parameters() if self.discriminative_lr == None else self.get_layer_wise_lr(*self.discriminative_lr), 
            lr=self.lr, weight_decay=self.wd)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log('train_loss', loss, on_step=False, on_epoch=True)
        
        self.training_step_outputs['loss'] += loss.detach().cpu()
        self.training_step_outputs['steps'] += 1
            
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log('val_loss', loss, on_step=False, on_epoch=True)

        self.validation_step_outputs['loss'] += loss.detach().cpu()
        self.validation_step_outputs['steps'] += 1
        
        return loss

    def on_train_epoch_end(self): 
        avg_loss = self.training_step_outputs['loss'] / self.training_step_outputs['steps']
        print(f"Average training loss for epoch {self.current_epoch}: {avg_loss.item():.4f}")
        self.training_step_outputs.clear() 

    def on_validation_epoch_end(self):
        avg_loss = self.validation_step_outputs['loss'] / self.validation_step_outputs['steps']
        print(f"Average validation loss for epoch {self.current_epoch}: {avg_loss.item():.4f}")
        self.validation_step_outputs.clear()

In [21]:
# Set random seed for reproducibility
pl.seed_everything(seed=42, workers=True)

transformacion_propuesta = transforms.Compose([
            transforms.Resize(256),

            # --- Geometric
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=.5),
            transforms.RandomVerticalFlip(p=.5),
            transforms.RandomRotation(degrees=90),

            # --- Color
            transforms.ColorJitter(
                brightness=0.1,   # simulate sunlight/shadow variation
                contrast=0.1,     # simulate different camera exposure
                saturation=0.1,   # account for differences in leaf pigmentation
                hue=0          # small hue shifts to avoid over-distorting
            ),
            transforms.RandomApply(
                [transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))],
                p=0.2
            ),
            
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

data_module = CassavaDataModule(cassava_df, IMAGE_ROOT_DIR)
data_module.setup('fit')

ckpt_path = "/home/alumno/Desktop/datos/Assigment 2/assigment2/best_valid_loss.ckpt"

base_model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

num_in_feat = base_model.fc.in_features
base_model.fc = nn.Linear(num_in_feat, 5)

# Como sabemos que el 
lightningModule_cargado = LightningModule.load_from_checkpoint(
    ckpt_path, 
    model=base_model 
)
model = lightningModule_cargado

# 3. Ponerlo en modo evaluación y enviarlo a la GPU si es posible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Seed set to 42


INFO: No se proporcionó 'train_transform'. Usando transformaciones de validación para train.
Dataset sizes: Train=14977, Val=3210, Test=3210
Train Transforms: Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Val/Test Transforms: Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


LightningModule(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [22]:
# Definir la transformación necesaria para la predicción (la misma de validación)
# Nota: LIME pasa imágenes tipo array uint8, necesitamos convertirlas a Tensor y normalizar
def get_preprocess_transform():
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

preprocess_transform = get_preprocess_transform()

def batch_predict(images):
    """
    Función requerida por LIME.
    Entrada: Lista o array de imágenes numpy (H, W, C)
    Salida: Array numpy con probabilidades (N_muestras, N_clases)
    """
    model.eval()
    batch_tensors = []
    
    for img in images:
        # LIME pasa las imágenes como numpy arrays. Se aplica el preprocesamiento.
        tensor_img = preprocess_transform(img)
        batch_tensors.append(tensor_img)
    
    # Crear batch
    batch = torch.stack(batch_tensors).to(device)
    
    with torch.no_grad():
        logits = model(batch)
        probs = F.softmax(logits, dim=1)
        
    return probs.cpu().numpy()

In [23]:
# Se crea la instancia del explicador
explainer = lime_image.LimeImageExplainer()

In [25]:
# 1. Instanciar el DataModule (si no lo has hecho ya en una celda anterior)
# Asegúrate de tener cassava_df e IMAGE_ROOT_DIR definidos
data_module = CassavaDataModule(cassava_df, IMAGE_ROOT_DIR, train_transform = transformacion_propuesta)

# 2. Ejecutar setup() para que se creen los datasets (train_ds, val_ds, test_ds)
data_module.setup()
# Obtenemos el dataset de test desde tu data_module
test_dataset = data_module.test_ds 
def find_examples_per_class(model, dataset, device, num_examples=10, num_classes=5):
    """
    Busca num_examples correctos y num_examples incorrectos para CADA clase.
    Retorna un diccionario con los índices.
    """
    # Estructura: results[clase]['correct'] = [indices...]
    results = {i: {'correct': [], 'incorrect': []} for i in range(num_classes)}
    
    # Contadores auxiliares para saber cuándo parar
    counts = {i: {'correct': 0, 'incorrect': 0} for i in range(num_classes)}
    
    model.eval()
    print(f"Buscando {num_examples} ejemplos correctos e incorrectos por clase...")
    
    total_imgs = len(dataset)
    
    for i in range(total_imgs):
        # Verificar si ya completamos TODAS las listas
        finished = True
        for c in range(num_classes):
            if counts[c]['correct'] < num_examples or counts[c]['incorrect'] < num_examples:
                finished = False
                break
        if finished:
            print(f"¡Búsqueda completada en el índice {i}!")
            break
            
        if i % 500 == 0:
            print(f"Escaneando imagen {i}/{total_imgs}...")

        # Inferencia
        img, label = dataset[i]
        
        # Si label viene como Tensor, lo convertimos a entero de Python
        if isinstance(label, torch.Tensor):
            label = label.item()
            
        img_tensor = img.unsqueeze(0).to(device)
        
        with torch.no_grad():
            logits = model(img_tensor)
            pred = torch.argmax(logits, dim=1).item()

        
        # Determinar si es correcto o incorrecto
        is_correct = (pred == label)
        tipo = 'correct' if is_correct else 'incorrect'
        
        # Guardar el índice si aún necesitamos ejemplos para esta clase (basada en Ground Truth) y tipo
        if counts[label][tipo] < num_examples:
            results[label][tipo].append(i)
            counts[label][tipo] += 1

    return results

# --- EJECUTAR LA BÚSQUEDA ---
examples_db = find_examples_per_class(model, test_dataset, device, num_examples=10)

# --- MOSTRAR RESUMEN ---
print("\n--- Resumen de Índices Encontrados ---")
for c in range(5):
    n_corr = len(examples_db[c]['correct'])
    n_inc = len(examples_db[c]['incorrect'])
    c_name = class_names[c] if 'class_names' in locals() else str(c)
    print(f"Clase {c} [{c_name}]:")
    print(f"   Correctos ({n_corr}): {examples_db[c]['correct']}")
    print(f"   Incorrectos ({n_inc}): {examples_db[c]['incorrect']}")

INFO: Usando 'train_transform' personalizada para el conjunto de entrenamiento.
Dataset sizes: Train=14977, Val=3210, Test=3210
Train Transforms: Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    RandomCrop(size=(224, 224), padding=None)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-90.0, 90.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=(0.9, 1.1), hue=None)
    RandomApply(
    p=0.2
    GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))
)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Val/Test Transforms: Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Buscando 10 ejemplos correctos e incorrectos por clase...
Escaneando imagen 0/3210...
¡

In [37]:
def get_original_image(dataset, idx):
    """
    Busca la ruta del archivo en el dataframe del dataset y carga la imagen original.
    """
    row = dataset.df.iloc[idx]
    img_name = row['image_id']
    # Accedemos al root_dir que guardaste en el dataset
    img_path = os.path.join(dataset.root_dir, img_name)
    
    # Cargamos y convertimos a numpy (H, W, 3)
    img = Image.open(img_path).convert('RGB')
    return np.array(img)
    
def explain_on_axis(img_numpy: np.ndarray, ax, type_pred: str, title: str ="Explicación"):
    """
    Genera la explicación LIME y la dibuja en el eje (ax) proporcionado.
    """
    print(f"   -> Procesando LIME para: {title}")
    
    # 1. Generar explicación (LIME tarda un poco aquí)
    explanation = explainer.explain_instance(
        img_numpy, 
        batch_predict, 
        top_labels=5,
        hide_color=0, 
        num_samples= 1000 # Puedes bajar esto a 500 para ir más rápido si solo pruebas
    )
    
    # 2. Obtener predicción y confianza
    probs = batch_predict([img_numpy])
    predicted_class = np.argmax(probs)
    confidence = np.max(probs)
    pred_name = class_names[predicted_class]
    
    # 3. Obtener imagen con máscara
    temp, mask = explanation.get_image_and_mask(
        predicted_class, 
        positive_only=False, 
        num_features=10, 
        hide_rest=False
    )

    # 4. DIBUJAR EN EL EJE (AX) EN LUGAR DE PLT.FIGURE
    ax.imshow(mark_boundaries(temp, mask))
    ax.set_title(f"{title}\nType of Pred {type_pred} Pred: {pred_name}\nConf: {confidence:.2f}", fontsize=10)
    ax.axis('off')

# 1. Función auxiliar para obtener la confianza de una imagen específica
def get_confidence_for_index(model, dataset, idx, device):
    img, _ = dataset[idx]
    img_tensor = img.unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1) # Convertir a probabilidad (0 a 1)
        confidence = torch.max(probs).item() # Tomar la probabilidad más alta
        
    return confidence

In [40]:
#GENERADOR NUEVO 
# CONFIGURACIÓN Y BUCLE PRINCIPAL
class_names = {
    0: "Cassava Bacterial Blight (CBB)",
    1: "Cassava Brown Streak Disease (CBSD)",
    2: "Cassava Green Mottle (CGM)",
    3: "Cassava Mosaic Disease (CMD)",
    4: "Healthy"
}

types_of_images = ['correct', 'incorrect']
NUM_TO_SHOW = 10   # 10 ejemplos por cada caso
print(f"Iniciando generación masiva de imágenes. Esto tomará tiempo...")

for key in class_names.keys():
    TARGET_CLASS = key
    nombre_clase = class_names.get(TARGET_CLASS)
    
    for tipo in types_of_images:
        print(f"Iniciando generación {tipo} clase {TARGET_CLASS}\n")
        # Lógica de ordenación solicitada
        if tipo == 'correct':
            ORDEN_DESCENDENTE = True  # Ver primero las de mayor confianza (las "mejores")
        else: 
            ORDEN_DESCENDENTE = False # Ver primero las de menor confianza (las "dudosas")

        # 1. Recuperar índices crudos
        raw_indices = examples_db[TARGET_CLASS][tipo]
        
        # Verificar si hay suficientes
        if not raw_indices:
            print(f"Saltando Clase {TARGET_CLASS} ({tipo}): No hay ejemplos encontrados.")
            continue

        print(f"\nProcesando: Clase {TARGET_CLASS} - {tipo} (Orden Descendente: {ORDEN_DESCENDENTE})")
        
        # 2. Calcular confianza y ORDENAR
        indices_with_conf = []
        for idx in raw_indices:
            conf = get_confidence_for_index(model, test_dataset, idx, device)
            indices_with_conf.append((idx, conf))
            
        # Ordenamos la lista de tuplas según la confianza (segundo elemento x[1])
        indices_with_conf.sort(key=lambda x: x[1], reverse=ORDEN_DESCENDENTE)
        
        # Nos quedamos con los top N índices
        final_indices = [x[0] for x in indices_with_conf][:NUM_TO_SHOW]

        if len(final_indices) == 0:
            continue

        # 3. Configuración de la figura (Grid)
        rows = len(final_indices) // 2 + (len(final_indices) % 2)
        cols = 2
        
        # Ajustamos altura dinámicamente según filas
        fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
        
        # Manejo de casos borde (si solo hay 1 fila o 1 imagen)
        if rows == 1 and cols == 1: axes = np.array([axes])
        elif rows == 1: axes = axes.reshape(1, -1) # Asegurar array 2D
        axes_flat = axes.flatten()
        
        # 4. Iterar y Pintar
        for i, idx_dataset in enumerate(final_indices):
            # Recuperar confianza para el título
            conf_val = next(c for id_, c in indices_with_conf if id_ == idx_dataset)
            
            # Obtener imagen
            img_numpy = get_original_image(test_dataset, idx_dataset)
            
            # Título informativo
            mytitle = f"Index: {idx_dataset} | Conf: {conf_val:.4f} | Tipo prediccion: {tipo.upper()}\n}"
            
            # Pintar
            current_ax = axes_flat[i]
            explain_on_axis(img_numpy, current_ax, type_pred = tipo, title=mytitle)
        
        # Apagar ejes vacíos si sobran huecos
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].axis('off')
        
        plt.suptitle(f"Clase: {nombre_clase} ({tipo})", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajuste para que no choque con el suptitle
        
        # 5. Guardar Imagen
        # Limpiamos el nombre de la clase de paréntesis o espacios raros para el archivo
        clean_name = nombre_clase.replace(" ", "_").replace("(", "").replace(")", "")
        filename = f"Grid_{tipo}_{clean_name}.png"
        
        plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0.1)
        print(f"--> Guardado: {filename}")
        
        # plt.show() # Descomenta si quieres verlas en pantalla también, pero consumirá mucha memoria
        plt.close(fig) # Importante cerrar la figura para liberar memoria RAM

Iniciando generación masiva de imágenes. Esto tomará tiempo...
Iniciando generación correct clase 0


Procesando: Clase 0 - correct (Orden Descendente: True)
   -> Procesando LIME para: Index: 52 | Conf: 0.6899
CORRECT...


  0%|          | 0/1000 [00:00<?, ?it/s]

   -> Procesando LIME para: Index: 299 | Conf: 0.6515
CORRECT...


  0%|          | 0/1000 [00:00<?, ?it/s]

   -> Procesando LIME para: Index: 102 | Conf: 0.6433
CORRECT...


...

   -> Procesando LIME para: Index: 19 | Conf: 0.7509
INCORRECT...


  0%|          | 0/1000 [00:00<?, ?it/s]

   -> Procesando LIME para: Index: 127 | Conf: 0.9806
INCORRECT...


  0%|          | 0/1000 [00:00<?, ?it/s]

--> Guardado: Grid_incorrect_Healthy.png
